## **Import the necessary libraries**
- we import the necessary libraries to be used in the dataset

In [ ]:
#import libraries
import os
import warnings
from pathlib import Path
import matplotlib
try:
    from IPython import get_ipython
    if get_ipython() is None or 'IPKernelApp' not in get_ipython().config:
        matplotlib.use('Agg')
except Exception:
    matplotlib.use('Agg')
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor, StackingClassifier, StackingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, classification_report, f1_score, mean_absolute_error, mean_squared_error, r2_score, silhouette_score, confusion_matrix, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from scipy.cluster.hierarchy import linkage, dendrogram

os.environ['LOKY_MAX_CPU_COUNT'] = '1'

PLOT_DIR = Path('generated_plots')
PLOT_DIR.mkdir(exist_ok=True)


# **Load the dataset**

- We load the dataset

In [75]:
#Load the dataset
df = pd.read_csv('global_graduate_employability_index.csv')
df.head()

,Country,Region,University_Name,Degree_Level,Field_of_Study,Graduation_Year,Employment_Rate_6_Months (%),Employment_Rate_12_Months (%),Average_Starting_Salary_USD,Top_Industry,Job_Role,Skill_1,Skill_2,Skill_3,Skill_Demand_Score (1–100),Remote_Work_Availability (%),Employer_Reputation_Score (1–100),Year
0,USA,North America,Harvard,Bachelor,Engineering,2017,79.3,85.6,66700,Manufacturing/Construction,Robotics Engineer,AutoCAD,Lean Six Sigma,MATLAB,69,8.8,66,2017
1,USA,North America,MIT,Bachelor,Engineering,2023,83.8,87.9,84500,Manufacturing/Construction,Civil Engineer,Lean Six Sigma,AutoCAD,MATLAB,71,65.4,63,2023
2,Israel,Middle East & Africa,Technion,Master,Healthcare & Medicine,2019,81.7,83.2,88300,Healthcare,Public Health Specialist,Clinical Research,Diagnostics,Patient Care,52,5.0,74,2019
3,India,Asia-Pacific,IIT Bombay,Master,Computer Science,2016,84.2,92.1,21000,Technology,AI Researcher,DevOps,Python,Cloud Computing,69,10.3,48,2016
4,South Africa,Middle East & Africa,University of Cape Town,PhD,Business & Finance,2023,83.6,86.3,48600,Finance/Consulting,Management Consultant,Financial Modeling,Data Analysis,Market Research,69,64.0,65,2023


## **Data Quality Checks**
 - Basic summarry statistics
 - Data types
 - Missing values
 - checking for outliers

In [76]:
#checking the shape of the dataset
df.shape
print(f'Number of rows in the dataset: {df.shape[0]}')
print(f'Number of columns in the dataset: {df.shape[1]}')

Number of rows in the dataset: 3500
Number of columns in the dataset: 18


The above output shows the number of rows and columns for the dataset. we have 18 variables, 15 independent variables and 3 dependent variables in the dataset

In [77]:
df.describe()

,Graduation_Year,Employment_Rate_6_Months (%),Employment_Rate_12_Months (%),Average_Starting_Salary_USD,Skill_Demand_Score (1–100),Remote_Work_Availability (%),Employer_Reputation_Score (1–100),Year
count,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000
mean,2020.021143,85.735486,90.409857,64668.000000,65.187429,40.060886,69.742286,2020.021143
std,3.198601,6.986723,6.743580,30727.510224,14.710632,30.318816,14.243037,3.198601
min,2015.000000,65.500000,68.000000,-1200.000000,30.000000,5.000000,40.000000,2015.000000
25%,2017.000000,80.700000,85.500000,39400.000000,55.000000,10.600000,60.000000,2017.000000
50%,2020.000000,85.600000,90.600000,63900.000000,64.000000,37.300000,70.000000,2020.000000
75%,2023.000000,90.900000,96.000000,85400.000000,75.000000,68.300000,79.250000,2023.000000
max,2025.000000,99.000000,100.000000,189400.000000,100.000000,90.000000,100.000000,2025.000000


In [78]:
df.value_counts()

Country       Region                University_Name                 Degree_Level  Field_of_Study         Graduation_Year  Employment_Rate_6_Months (%)  Employment_Rate_12_Months (%)  Average_Starting_Salary_USD  Top_Industry                Job_Role                  Skill_1             Skill_2               Skill_3          Skill_Demand_Score (1–100)  Remote_Work_Availability (%)  Employer_Reputation_Score (1–100)  Year
USA           North America         Harvard                         Bachelor      Engineering            2017             79.3                          85.6                           66700                        Manufacturing/Construction  Robotics Engineer         AutoCAD             Lean Six Sigma        MATLAB           69                          8.8                           66                                 2017    1
                                    MIT                             Bachelor      Engineering            2023             83.8                       

In [79]:
#checking for missing values
df.isnull().sum()

Country                              0
Region                               0
University_Name                      0
Degree_Level                         0
Field_of_Study                       0
Graduation_Year                      0
Employment_Rate_6_Months (%)         0
Employment_Rate_12_Months (%)        0
Average_Starting_Salary_USD          0
Top_Industry                         0
Job_Role                             0
Skill_1                              0
Skill_2                              0
Skill_3                              0
Skill_Demand_Score (1–100)           0
Remote_Work_Availability (%)         0
Employer_Reputation_Score (1–100)    0
Year                                 0
dtype: int64

We don't have missing values in our dataset.

In [80]:
# Identify target columns
targets = ['Average_Starting_Salary_USD', 'Employment_Rate_6_Months (%)', 'Employment_Rate_12_Months (%)']

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot histograms for each target
for i, col in enumerate(targets):
    sns.histplot(df[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_9140\248262778.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [81]:
#check information about the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 18 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Country                            3500 non-null   str    
 1   Region                             3500 non-null   str    
 2   University_Name                    3500 non-null   str    
 3   Degree_Level                       3500 non-null   str    
 4   Field_of_Study                     3500 non-null   str    
 5   Graduation_Year                    3500 non-null   int64  
 6   Employment_Rate_6_Months (%)       3500 non-null   float64
 7   Employment_Rate_12_Months (%)      3500 non-null   float64
 8   Average_Starting_Salary_USD        3500 non-null   int64  
 9   Top_Industry                       3500 non-null   str    
 10  Job_Role                           3500 non-null   str    
 11  Skill_1                            3500 non-null   str    
 12  Ski

- **Checking for duplicates**

In [82]:
#Count total duplicate rows across all columns
duplicates_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates_count}")

#To see the actual duplicate rows (if any existed)
if duplicates_count > 0:
    duplicate_rows = df[df.duplicated()]
    print(duplicate_rows)

#Check for duplicates in specific columns (e.g., University_Name and Graduation_Year)
#This is useful if a student shouldn't appear twice for the same year/university
specific_duplicates = df.duplicated(subset=['University_Name', 'Graduation_Year', 'Field_of_Study']).sum()
print(f"Duplicates based on specific columns: {specific_duplicates}")

Total duplicate rows: 0
Duplicates based on specific columns: 1522


In [83]:
# View duplicates based on specific columns
subset_duplicates = df[df.duplicated(subset=['University_Name', 'Graduation_Year'], keep=False)]
print(subset_duplicates)

           Country                Region                 University_Name  \
0              USA         North America                         Harvard   
1              USA         North America                             MIT   
2           Israel  Middle East & Africa                        Technion   
3            India          Asia-Pacific                      IIT Bombay   
4     South Africa  Middle East & Africa         University of Cape Town   
...            ...                   ...                             ...   
3495        Israel  Middle East & Africa  Hebrew University of Jerusalem   
3496         Japan          Asia-Pacific                Kyoto University   
3497     Australia          Asia-Pacific            University of Sydney   
3498           UAE  Middle East & Africa              Khalifa University   
3499   Switzerland                Europe                            EPFL   

     Degree_Level         Field_of_Study  Graduation_Year  \
0        Bachelor         

In [84]:
# Finding rows where University, Degree, and Graduation Year are the same
subset_cols = ['University_Name', 'Degree_Level', 'Graduation_Year']
subset_duplicates = df[df.duplicated(subset=subset_cols, keep=False)]

print(f"Showing rows with same University, Degree, and Year:")
print(subset_duplicates.sort_values(by=subset_cols).head(10))

Showing rows with same University, Degree, and Year:
        Country        Region University_Name Degree_Level     Field_of_Study  \
520   Australia  Asia-Pacific             ANU     Bachelor    Social Sciences   
1842  Australia  Asia-Pacific             ANU     Bachelor  Data Science & AI   
3354  Australia  Asia-Pacific             ANU     Bachelor    Social Sciences   
3009  Australia  Asia-Pacific             ANU     Bachelor        Engineering   
3026  Australia  Asia-Pacific             ANU     Bachelor    Social Sciences   
2437  Australia  Asia-Pacific             ANU     Bachelor        Engineering   
2441  Australia  Asia-Pacific             ANU     Bachelor   Computer Science   
741   Australia  Asia-Pacific             ANU     Bachelor   Natural Sciences   
1613  Australia  Asia-Pacific             ANU     Bachelor        Engineering   
2302  Australia  Asia-Pacific             ANU     Bachelor        Engineering   

      Graduation_Year  Employment_Rate_6_Months (%)  \


In [85]:
# Statistical summary of numerical columns
print(df.describe())

# Check how many graduates are in each Field of Study
print(df['Field_of_Study'].value_counts())

# Check Average Salary by Degree Level
print(df.groupby('Degree_Level')['Average_Starting_Salary_USD'].mean())

       Graduation_Year  Employment_Rate_6_Months (%)  \
count      3500.000000                   3500.000000   
mean       2020.021143                     85.735486   
std           3.198601                      6.986723   
min        2015.000000                     65.500000   
25%        2017.000000                     80.700000   
50%        2020.000000                     85.600000   
75%        2023.000000                     90.900000   
max        2025.000000                     99.000000   

       Employment_Rate_12_Months (%)  Average_Starting_Salary_USD  \
count                    3500.000000                  3500.000000   
mean                       90.409857                 64668.000000   
std                         6.743580                 30727.510224   
min                        68.000000                 -1200.000000   
25%                        85.500000                 39400.000000   
50%                        90.600000                 63900.000000   
75%         

In [86]:
# Check for duplicate ROWS
# duplicated() returns a boolean Series indicating whether each row is a duplicate
duplicate_rows_count = df.duplicated().sum()
duplicate_rows = df[df.duplicated()]

print(f"Total Duplicate Rows: {duplicate_rows_count}")

#Check for duplicate COLUMNS
# A function to compare every column with every other column to find identical values
def get_duplicate_columns(df):
    duplicate_columns = []
    for i in range(len(df.columns)):
        col1 = df.iloc[:, i]
        for j in range(i + 1, len(df.columns)):
            col2 = df.iloc[:, j]
            # If all values in both columns are equal
            if col1.equals(col2):
                duplicate_columns.append((df.columns[i], df.columns[j]))
    return duplicate_columns

duplicate_cols = get_duplicate_columns(df)

if duplicate_cols:
    print("\nDuplicate Column Pairs Found:")
    for col_pair in duplicate_cols:
        print(f" - '{col_pair[1]}' is identical to '{col_pair[0]}'")
else:
    print("\nNo duplicate columns found.")

Total Duplicate Rows: 0

Duplicate Column Pairs Found:
 - 'Year' is identical to 'Graduation_Year'


In [87]:
# Displaying the first 10 rows of the identical columns for comparison
print("Comparing 'Graduation_Year' and 'Year':")
print(df[['Graduation_Year', 'Year']].head(10))

# Confirm they are 100% identical
are_identical = df['Graduation_Year'].equals(df['Year'])
print(f"\nAre the columns identical? {are_identical}")

Comparing 'Graduation_Year' and 'Year':
   Graduation_Year  Year
0             2017  2017
1             2023  2023
2             2019  2019
3             2016  2016
4             2023  2023
5             2015  2015
6             2025  2025
7             2025  2025
8             2023  2023
9             2016  2016

Are the columns identical? True


- **Checking for outliers**

In [88]:
#Select numerical columns
num_cols = df.select_dtypes(include=['number']).columns

#Calculate outliers and display summary
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)
    
    #Identify outliers
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Column: {col} | Outlier Count: {len(outliers)}")

#Generate Box Plots to visualize outliers
plt.figure(figsize=(15, 10))
for i, col in enumerate(num_cols, 1):
    plt.subplot(3, 3, i)
    sns.boxplot(x=df[col], color='skyblue')
    plt.title(f'Outliers in {col}')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'outlier_plots.png')

Column: Graduation_Year | Outlier Count: 0
Column: Employment_Rate_6_Months (%) | Outlier Count: 0
Column: Employment_Rate_12_Months (%) | Outlier Count: 4
Column: Average_Starting_Salary_USD | Outlier Count: 7
Column: Skill_Demand_Score (1–100) | Outlier Count: 0
Column: Remote_Work_Availability (%) | Outlier Count: 0
Column: Employer_Reputation_Score (1–100) | Outlier Count: 0
Column: Year | Outlier Count: 0


## **Data Cleaning**
  - Handling outliers

In [89]:
# 1. Clean column names (remove any trailing spaces)
df.columns = df.columns.str.strip()

# Identify numerical columns for scaling later
num_cols = df.select_dtypes(include=['number']).columns.tolist()

# 2. Handle Outliers via Capping (Winsorization)
# Targeted columns based on the outlier analysis
cols_to_treat = ['Employment_Rate_12_Months (%)', 'Average_Starting_Salary_USD']

# Store a copy for visualization comparison
df_original = df.copy()

for col in cols_to_treat:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)
    
    # Capping: Values outside the bounds are set to the bound value
    # This removes the extreme "pull" of outliers without deleting data rows
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    print(f"Outliers capped for: {col}")

# 3. Feature Scaling (Standardization)
# Most ML models require features on the same scale
scaler = StandardScaler()
df_processed = df.copy()

# Fit and transform the numerical columns
df_processed[num_cols] = scaler.fit_transform(df[num_cols])

# 4. Visualization: Before vs After Treatment (Salary and Employment Rate)
# Creating a 2x2 grid to show both columns
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Row 1: Average Starting Salary
sns.boxplot(data=df_original, x='Average_Starting_Salary_USD', ax=axes[0, 0], color='salmon')
axes[0, 0].set_title('Original Salary (With Outliers)')

sns.boxplot(data=df, x='Average_Starting_Salary_USD', ax=axes[0, 1], color='skyblue')
axes[0, 1].set_title('Salary After Capping (Outliers Tamed)')

# Row 2: Employment Rate 12 Months
sns.boxplot(data=df_original, x='Employment_Rate_12_Months (%)', ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Original Employment Rate (With Outliers)')

sns.boxplot(data=df, x='Employment_Rate_12_Months (%)', ax=axes[1, 1], color='gold')
axes[1, 1].set_title('Employment Rate After Capping (Outliers Tamed)')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'outlier_treatment_comparison.png')
plt.show()

# Save the final cleaned and scaled data for model training
df_processed.to_csv('processed_graduate_employability.csv', index=False)

Outliers capped for: Employment_Rate_12_Months (%)
Outliers capped for: Average_Starting_Salary_USD


C:\Users\user\AppData\Local\Temp\ipykernel_9140\701965029.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


- Dropping duplicates

In [90]:
# Dropping a single column
df = df.drop('Year', axis=1, errors='ignore')

print(df.columns.tolist())

['Country', 'Region', 'University_Name', 'Degree_Level', 'Field_of_Study', 'Graduation_Year', 'Employment_Rate_6_Months (%)', 'Employment_Rate_12_Months (%)', 'Average_Starting_Salary_USD', 'Top_Industry', 'Job_Role', 'Skill_1', 'Skill_2', 'Skill_3', 'Skill_Demand_Score (1–100)', 'Remote_Work_Availability (%)', 'Employer_Reputation_Score (1–100)']


In [91]:
df.head(10)

,Country,Region,University_Name,Degree_Level,Field_of_Study,Graduation_Year,Employment_Rate_6_Months (%),Employment_Rate_12_Months (%),Average_Starting_Salary_USD,Top_Industry,Job_Role,Skill_1,Skill_2,Skill_3,Skill_Demand_Score (1–100),Remote_Work_Availability (%),Employer_Reputation_Score (1–100)
0,USA,North America,Harvard,Bachelor,Engineering,2017,79.3,85.6,66700,Manufacturing/Construction,Robotics Engineer,AutoCAD,Lean Six Sigma,MATLAB,69,8.8,66
1,USA,North America,MIT,Bachelor,Engineering,2023,83.8,87.9,84500,Manufacturing/Construction,Civil Engineer,Lean Six Sigma,AutoCAD,MATLAB,71,65.4,63
2,Israel,Middle East & Africa,Technion,Master,Healthcare & Medicine,2019,81.7,83.2,88300,Healthcare,Public Health Specialist,Clinical Research,Diagnostics,Patient Care,52,5.0,74
3,India,Asia-Pacific,IIT Bombay,Master,Computer Science,2016,84.2,92.1,21000,Technology,AI Researcher,DevOps,Python,Cloud Computing,69,10.3,48
4,South Africa,Middle East & Africa,University of Cape Town,PhD,Business & Finance,2023,83.6,86.3,48600,Finance/Consulting,Management Consultant,Financial Modeling,Data Analysis,Market Research,69,64.0,65
5,South Africa,Middle East & Africa,University of Cape Town,PhD,Computer Science,2015,99.0,100.0,39200,Technology,Software Engineer,React,Java,Cloud Computing,59,14.1,51
6,UK,Europe,LSE,Master,Business & Finance,2025,71.5,73.8,86000,Finance/Consulting,Management Consultant,Excel,Strategic Planning,Data Analysis,71,90.0,72
7,Sweden,Europe,Karolinska Institute,PhD,Social Sciences,2025,81.5,83.5,79800,Government/NGO,Economist,Public Policy,Communication,Data Analysis,62,85.4,85
8,Netherlands,Europe,University of Amsterdam,Master,Social Sciences,2023,72.4,78.1,60900,Government/NGO,Policy Analyst,Qualitative Research,Communication,Data Analysis,62,63.2,79
9,UK,Europe,Imperial College London,Master,Data Science & AI,2016,95.9,99.2,78800,Technology/Consulting,Data Analyst,Deep Learning,Big Data,R,68,11.7,84


## **Exploratory Data Analysis**

In [92]:
#Correlation Heatmap
#This helps identify which variables impact employability most.
plt.figure(figsize=(12, 8))
numeric_df = df.select_dtypes(include=['number'])
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Numerical Features')
plt.savefig(PLOT_DIR / 'correlation_heatmap.png')

#Visualizing Key Drivers (Fixing the deprecation warning)
#We assign x to hue to use the 'viridis' palette without a warning.
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=df, 
    x='Field_of_Study', 
    y='Average_Starting_Salary_USD', 
    hue='Field_of_Study', 
    palette='viridis'
)

#Removing the legend since each bar is already labeled on the X-axis
if ax.get_legend():
    ax.get_legend().remove()

plt.xticks(rotation=45)
plt.title('Average Starting Salary by Field of Study')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'salary_by_field.png')

#Target Creation for Classification
#As per the project plan, defining 'High_Employability' as a binary target.
threshold = df['Employment_Rate_12_Months (%)'].median()
df['High_Employability'] = (df['Employment_Rate_12_Months (%)'] > threshold).astype(int)

## Dropping leakage columns

In [93]:
#DROP LEAKAGE COLUMNS FIRST 
#These correlate highly but are "cheating" variables (outcomes)
leakage_cols = ['Employment_Rate_6_Months (%)', 'Average_Starting_Salary_USD', 'Top_Industry', 'Job_Role', 'Graduation_Year', 'Year', 'University_Name', 'Country', 'Region']
df = df.drop(columns=leakage_cols, errors='ignore')

#CALCULATE CORRELATION SAFELY
#We use 'numeric_only=True' to avoid the ValueError you encountered
target = 'Employment_Rate_12_Months (%)'
correlations = df.corr(numeric_only=True)[target].abs().sort_values(ascending=False)

#IDENTIFY COLUMNS TO DROP
#We define "low correlation" as any value less than 0.1
threshold = 0.1
low_corr_cols = correlations[correlations < threshold].index.tolist()

print("Columns showing weak correlation (< 0.1) and will be dropped:")
print(low_corr_cols)
#DROP THE LOW CORRELATION COLUMNS
df = df.drop(columns=low_corr_cols)

#Final check of the remaining features
print("\nFinal set of features for training:")
print(df.columns.tolist())

Columns showing weak correlation (< 0.1) and will be dropped:
['Remote_Work_Availability (%)', 'Employer_Reputation_Score (1–100)']

Final set of features for training:
['Degree_Level', 'Field_of_Study', 'Employment_Rate_12_Months (%)', 'Skill_1', 'Skill_2', 'Skill_3', 'Skill_Demand_Score (1–100)', 'High_Employability']


- ## **Encoding**

In [94]:
#Ordinal Encoding for Degree (Mapping to 1, 2, 3)
degree_map = {'Bachelor': 1, 'Master': 2, 'PhD': 3}
if df['Degree_Level'].dtype == 'object':
    df['Degree_Level'] = df['Degree_Level'].map(degree_map)

#Label Encoding for Skills (High-Cardinality Strings to IDs)
#We use Label Encoding here instead of One-Hot to avoid creating hundreds of columns
le = LabelEncoder()
for col in ['Skill_1', 'Skill_2', 'Skill_3']:
    df[col] = le.fit_transform(df[col].astype(str))

#One-Hot Encoding for Field_of_Study (Nominal)
#Using 'drop_first=True' prevents the "Dummy Variable Trap"
df = pd.get_dummies(df, columns=['Field_of_Study'], drop_first=True, dtype=int)

#Feature Scaling for Skill Demand Score
#Scaling ensures this 1–100 range doesn't bias the Logistic Regression model
scaler = StandardScaler()
df['Skill_Demand_Score (1–100)'] = scaler.fit_transform(df[['Skill_Demand_Score (1–100)']])

#Check the Final Preprocessed Results
print("Final Encoded and Scaled Columns:")
print(df.columns.tolist())
print("\nFirst 5 rows of the Final Preprocessed Dataset:")
print(df.head())

Final Encoded and Scaled Columns:
['Degree_Level', 'Employment_Rate_12_Months (%)', 'Skill_1', 'Skill_2', 'Skill_3', 'Skill_Demand_Score (1–100)', 'High_Employability', 'Field_of_Study_Computer Science', 'Field_of_Study_Data Science & AI', 'Field_of_Study_Engineering', 'Field_of_Study_Healthcare & Medicine', 'Field_of_Study_Natural Sciences', 'Field_of_Study_Social Sciences']

First 5 rows of the Final Preprocessed Dataset:
  Degree_Level  Employment_Rate_12_Months (%)  Skill_1  Skill_2  Skill_3  \
0     Bachelor                           85.6        0       16       17   
1     Bachelor                           87.9       16        0       17   
2       Master                           83.2        2       10       22   
3       Master                           92.1        9       25        3   
4          PhD                           86.3       13        6       19   

   Skill_Demand_Score (1–100)  High_Employability  \
0                    0.259208                   0   
1        

In [95]:
df.head()

,Degree_Level,Employment_Rate_12_Months (%),Skill_1,Skill_2,Skill_3,Skill_Demand_Score (1–100),High_Employability,Field_of_Study_Computer Science,Field_of_Study_Data Science & AI,Field_of_Study_Engineering,Field_of_Study_Healthcare & Medicine,Field_of_Study_Natural Sciences,Field_of_Study_Social Sciences
0,Bachelor,85.6,0,16,17,0.259208,0,0,0,1,0,0,0
1,Bachelor,87.9,16,0,17,0.395184,0,0,0,1,0,0,0
2,Master,83.2,2,10,22,-0.896584,0,0,0,0,1,0,0
3,Master,92.1,9,25,3,0.259208,1,1,0,0,0,0,0
4,PhD,86.3,13,6,19,0.259208,0,0,0,0,0,0,0


## **Models**

This section uses the cleaned dataset to build prediction models without changing the original cleaning steps above. The code creates extra modeling features, evaluates classification models for unemployability risk, regression models for 12-month employment rate, and an unsupervised clustering model, then saves the results and visualizations.

In [96]:
# Modeling section.
RANDOM_STATE = 42
SOURCE_FILE = 'global_graduate_employability_index.csv'
CLASSIFICATION_RESULTS_FILE = 'classification_model_results.csv'
REGRESSION_RESULTS_FILE = 'regression_model_results.csv'
CLUSTER_RESULTS_FILE = 'risk_cluster_profile.csv'

def load_and_preserve_cleaning(path):
    model_df = pd.read_csv(path).copy()
    model_df.columns = model_df.columns.str.strip()
    model_df = model_df.drop(columns=['Year'], errors='ignore')

    for col in ['Employment_Rate_12_Months (%)', 'Average_Starting_Salary_USD']:
        q1 = model_df[col].quantile(0.25)
        q3 = model_df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        upper_bound = q3 + (1.5 * iqr)
        model_df[col] = model_df[col].clip(lower=lower_bound, upper=upper_bound)

    return model_df

def add_modeling_features(model_df):
    featured = model_df.copy()
    degree_map = {'Bachelor': 1, 'Master': 2, 'PhD': 3}
    featured['Degree_Level_Ordinal'] = featured['Degree_Level'].map(degree_map)
    featured['Graduation_Recency'] = featured['Graduation_Year'] - featured['Graduation_Year'].min()
    featured['Demand_x_Reputation'] = featured['Skill_Demand_Score (1–100)'] * featured['Employer_Reputation_Score (1–100)']
    featured['Demand_x_Remote'] = featured['Skill_Demand_Score (1–100)'] * featured['Remote_Work_Availability (%)']
    featured['Reputation_x_Remote'] = featured['Employer_Reputation_Score (1–100)'] * featured['Remote_Work_Availability (%)']
    featured['Skills_Profile'] = featured['Skill_1'].astype(str) + ' | ' + featured['Skill_2'].astype(str) + ' | ' + featured['Skill_3'].astype(str)
    return featured

def build_feature_table(model_df):
    target_regression = model_df['Employment_Rate_12_Months (%)'].copy()
    risk_cutoff = float(target_regression.quantile(0.25))
    target_classification = (target_regression <= risk_cutoff).astype(int)

    leakage_columns = [
        'Employment_Rate_12_Months (%)',
        'Employment_Rate_6_Months (%)',
        'Average_Starting_Salary_USD',
        'Top_Industry',
        'Job_Role',
        'University_Name'
    ]

    feature_table = model_df.drop(columns=leakage_columns, errors='ignore').copy()
    return feature_table, target_classification, target_regression, risk_cutoff

def build_preprocessor(X):
    categorical_cols = X.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()
    numeric_cols = [col for col in X.columns if col not in categorical_cols]

    try:
        onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown='ignore', sparse=False)

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', onehot)
    ])

    return ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

def evaluate_classification_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    models = {
        'Logistic Regression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=1),
        'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
    }

    rows = []
    for model_name, estimator in models.items():
        pipeline = Pipeline(steps=[('preprocessor', build_preprocessor(X_train)), ('model', estimator)])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        scores = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, 'predict_proba') else pipeline.decision_function(X_test)

        rows.append({
            'Model': model_name,
            'Accuracy': accuracy_score(y_test, predictions),
            'Precision': precision_score(y_test, predictions, zero_division=0),
            'Recall': recall_score(y_test, predictions, zero_division=0),
            'F1': f1_score(y_test, predictions, zero_division=0),
            'ROC_AUC': roc_auc_score(y_test, scores)
        })

    return pd.DataFrame(rows).sort_values(by=['ROC_AUC', 'F1', 'Recall'], ascending=False).reset_index(drop=True)

def evaluate_regression_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

    models = {
        'Ridge Regression': Ridge(alpha=1.0),
        'Random Forest Regressor': RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1),
        'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=RANDOM_STATE)
    }

    rows = []
    for model_name, estimator in models.items():
        pipeline = Pipeline(steps=[('preprocessor', build_preprocessor(X_train)), ('model', estimator)])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)

        rows.append({
            'Model': model_name,
            'R2': r2_score(y_test, predictions),
            'RMSE': np.sqrt(mean_squared_error(y_test, predictions)),
            'MAE': mean_absolute_error(y_test, predictions)
        })

    return pd.DataFrame(rows).sort_values(by=['R2', 'RMSE'], ascending=[False, True]).reset_index(drop=True)

def evaluate_clustering_model(X, y_risk, y_rate):
    transformed = build_preprocessor(X).fit_transform(X)

    best_score = -np.inf
    best_labels = None
    best_k = None

    for n_clusters in range(2, 6):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=RANDOM_STATE)
            labels = kmeans.fit_predict(transformed)
            score = silhouette_score(transformed, labels)
        if score > best_score:
            best_score = score
            best_labels = labels
            best_k = n_clusters

    cluster_profile = pd.DataFrame({
        'Cluster': best_labels,
        'Employment_Rate_12_Months (%)': y_rate.values,
        'Unemployability_Risk': y_risk.values
    }).groupby('Cluster', as_index=False).agg(
        Count=('Cluster', 'size'),
        Average_Employment_Rate=('Employment_Rate_12_Months (%)', 'mean'),
        Risk_Rate=('Unemployability_Risk', 'mean')
    ).sort_values(by='Average_Employment_Rate').reset_index(drop=True)

    cluster_profile['Silhouette_Score'] = round(best_score, 4)
    cluster_profile['Selected_K'] = best_k
    return cluster_profile

def plot_classification_results(classification_results):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(data=classification_results, x='ROC_AUC', y='Model', hue='Model', palette='Blues_r', legend=False, ax=axes[0])
    axes[0].set_title('Classification Models by ROC-AUC')
    axes[0].set_xlabel('ROC-AUC')
    axes[0].set_ylabel('Model')

    classification_long = classification_results.melt(id_vars='Model', value_vars=['Accuracy', 'Precision', 'Recall', 'F1'], var_name='Metric', value_name='Score')
    sns.barplot(data=classification_long, x='Score', y='Model', hue='Metric', ax=axes[1])
    axes[1].set_title('Classification Metric Comparison')
    axes[1].set_xlabel('Score')
    axes[1].set_ylabel('Model')
    axes[1].legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')

    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'classification_model_performance.png', bbox_inches='tight')
    plt.show()

def plot_regression_results(regression_results):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(data=regression_results, x='R2', y='Model', hue='Model', palette='Greens_r', legend=False, ax=axes[0])
    axes[0].set_title('Regression Models by R-squared')
    axes[0].set_xlabel('R-squared')
    axes[0].set_ylabel('Model')

    regression_long = regression_results.melt(id_vars='Model', value_vars=['RMSE', 'MAE'], var_name='Metric', value_name='Score')
    sns.barplot(data=regression_long, x='Score', y='Model', hue='Metric', ax=axes[1])
    axes[1].set_title('Regression Error Comparison')
    axes[1].set_xlabel('Error')
    axes[1].set_ylabel('Model')
    axes[1].legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')

    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'regression_model_performance.png', bbox_inches='tight')
    plt.show()

def plot_cluster_profile(cluster_profile):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(data=cluster_profile, x='Cluster', y='Average_Employment_Rate', hue='Cluster', palette='Oranges', legend=False, ax=axes[0])
    axes[0].set_title('Cluster Average Employment Rate')
    axes[0].set_xlabel('Cluster')
    axes[0].set_ylabel('Average Employment Rate')

    sns.barplot(data=cluster_profile, x='Cluster', y='Risk_Rate', hue='Cluster', palette='Reds', legend=False, ax=axes[1])
    axes[1].set_title('Cluster Unemployability Risk Rate')
    axes[1].set_xlabel('Cluster')
    axes[1].set_ylabel('Risk Rate')

    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'cluster_profile.png', bbox_inches='tight')
    plt.show()

model_df = load_and_preserve_cleaning(SOURCE_FILE)
model_df = add_modeling_features(model_df)
X, y_classification, y_regression, risk_cutoff = build_feature_table(model_df)

classification_results = evaluate_classification_models(X, y_classification)
regression_results = evaluate_regression_models(X, y_regression)
cluster_profile = evaluate_clustering_model(X, y_classification, y_regression)

classification_results.to_csv(CLASSIFICATION_RESULTS_FILE, index=False)
regression_results.to_csv(REGRESSION_RESULTS_FILE, index=False)
cluster_profile.to_csv(CLUSTER_RESULTS_FILE, index=False)

print(f'Binary unemployability risk is defined as Employment_Rate_12_Months (%) <= {risk_cutoff:.2f}.')
print('\nClassification models ranked by ROC-AUC:')
print(classification_results.round(4).to_string(index=False))
print('\nRegression models ranked by R^2:')
print(regression_results.round(4).to_string(index=False))
print('\nUnsupervised risk segmentation using KMeans:')
print(cluster_profile.round(4).to_string(index=False))
print(f'\nSaved results to {CLASSIFICATION_RESULTS_FILE}, {REGRESSION_RESULTS_FILE}, and {CLUSTER_RESULTS_FILE}.')

plot_classification_results(classification_results)
plot_regression_results(regression_results)
plot_cluster_profile(cluster_profile)

classification_results


Binary unemployability risk is defined as Employment_Rate_12_Months (%) <= 85.50.

Classification models ranked by ROC-AUC:
              Model  Accuracy  Precision  Recall     F1  ROC_AUC
Logistic Regression    0.7071     0.4551  0.8686 0.5972   0.8008
  Gradient Boosting    0.7429     0.4699  0.2229 0.3023   0.7924
      Random Forest    0.6814     0.4281  0.8171 0.5619   0.7823

Regression models ranked by R^2:
                      Model     R2   RMSE    MAE
Gradient Boosting Regressor 0.4517 4.6703 3.7183
    Random Forest Regressor 0.4262 4.7776 3.8056
           Ridge Regression 0.4018 4.8785 3.8384

Unsupervised risk segmentation using KMeans:
 Cluster  Count  Average_Employment_Rate  Risk_Rate  Silhouette_Score  Selected_K
       0   1830                  90.3275     0.2601            0.2181           2
       1   1670                  90.5033     0.2395            0.2181           2

Saved results to classification_model_results.csv, regression_model_results.csv, and risk_clu

C:\Users\user\AppData\Local\Temp\ipykernel_9140\979851837.py:173: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_9140\979851837.py:191: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_9140\979851837.py:207: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.707143,0.455090,0.868571,0.597250,0.800827
1,Gradient Boosting,0.742857,0.469880,0.222857,0.302326,0.792414
2,Random Forest,0.681429,0.428144,0.817143,0.561886,0.782335


## **Model Visualizations**

This section shows the model results inside the notebook. It displays classification and regression evaluation plots, and also adds clustering visuals such as the elbow method, PCA cluster scatter plot, dendrogram, and cluster profile so the model behavior is easier to interpret.

In [97]:
# Extra model visualizations for inline notebook display.
classification_model_map = {
    'Logistic Regression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}
regression_model_map = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=RANDOM_STATE)
}

best_classification_name = classification_results.iloc[0]['Model']
best_regression_name = regression_results.iloc[0]['Model']

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X, y_classification, test_size=0.2, random_state=RANDOM_STATE, stratify=y_classification)
best_cls_pipeline = Pipeline(steps=[('preprocessor', build_preprocessor(X_train_cls)), ('model', classification_model_map[best_classification_name])])
best_cls_pipeline.fit(X_train_cls, y_train_cls)
y_pred_cls = best_cls_pipeline.predict(X_test_cls)
y_score_cls = best_cls_pipeline.predict_proba(X_test_cls)[:, 1] if hasattr(best_cls_pipeline, 'predict_proba') else best_cls_pipeline.decision_function(X_test_cls)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_regression, test_size=0.2, random_state=RANDOM_STATE)
best_reg_pipeline = Pipeline(steps=[('preprocessor', build_preprocessor(X_train_reg)), ('model', regression_model_map[best_regression_name])])
best_reg_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg = best_reg_pipeline.predict(X_test_reg)
reg_residuals = y_test_reg - y_pred_reg

transformed = build_preprocessor(X).fit_transform(X)
elbow_rows = []
best_k = None
best_score = -np.inf
best_labels = None
for k in range(1, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = km.fit_predict(transformed)
    elbow_rows.append({'K': k, 'Inertia': km.inertia_})
    if k > 1:
        sil = silhouette_score(transformed, labels)
        if sil > best_score:
            best_score = sil
            best_k = k
            best_labels = labels

cluster_coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(transformed)
cluster_scatter = pd.DataFrame({'PC1': cluster_coords[:, 0], 'PC2': cluster_coords[:, 1], 'Cluster': best_labels.astype(str), 'Unemployability_Risk': y_classification.values})
sample_size = min(200, transformed.shape[0])
sample_idx = np.random.default_rng(RANDOM_STATE).choice(np.arange(transformed.shape[0]), size=sample_size, replace=False)
linkage_matrix = linkage(transformed[sample_idx], method='ward')

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.barplot(data=classification_results, x='ROC_AUC', y='Model', hue='Model', palette='Blues_r', legend=False, ax=axes[0, 0])
axes[0, 0].set_title('Classification Models by ROC-AUC')
classification_long = classification_results.melt(id_vars='Model', value_vars=['Accuracy', 'Precision', 'Recall', 'F1'], var_name='Metric', value_name='Score')
sns.barplot(data=classification_long, x='Score', y='Model', hue='Metric', ax=axes[0, 1])
axes[0, 1].set_title('Classification Metric Comparison')
fpr, tpr, _ = roc_curve(y_test_cls, y_score_cls)
axes[1, 0].plot(fpr, tpr, label=f'{best_classification_name} ROC')
axes[1, 0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1, 0].set_title('ROC Curve')
axes[1, 0].legend()
sns.heatmap(confusion_matrix(y_test_cls, y_pred_cls), annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1, 1])
axes[1, 1].set_title(f'Confusion Matrix: {best_classification_name}')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'classification_model_performance.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.barplot(data=regression_results, x='R2', y='Model', hue='Model', palette='Greens_r', legend=False, ax=axes[0, 0])
axes[0, 0].set_title('Regression Models by R-squared')
regression_long = regression_results.melt(id_vars='Model', value_vars=['RMSE', 'MAE'], var_name='Metric', value_name='Score')
sns.barplot(data=regression_long, x='Score', y='Model', hue='Metric', ax=axes[0, 1])
axes[0, 1].set_title('Regression Error Comparison')
axes[1, 0].scatter(y_test_reg, y_pred_reg, alpha=0.6, color='darkgreen')
axes[1, 0].plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], linestyle='--', color='black')
axes[1, 0].set_title(f'Actual vs Predicted: {best_regression_name}')
axes[1, 1].scatter(y_pred_reg, reg_residuals, alpha=0.6, color='firebrick')
axes[1, 1].axhline(0, linestyle='--', color='black')
axes[1, 1].set_title(f'Residual Plot: {best_regression_name}')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'regression_model_performance.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.lineplot(data=pd.DataFrame(elbow_rows), x='K', y='Inertia', marker='o', ax=axes[0, 0], color='darkorange')
axes[0, 0].axvline(best_k, linestyle='--', color='gray')
axes[0, 0].set_title('Elbow Method for KMeans')
sns.scatterplot(data=cluster_scatter, x='PC1', y='PC2', hue='Cluster', style='Unemployability_Risk', palette='tab10', alpha=0.7, ax=axes[0, 1])
axes[0, 1].set_title('Cluster Scatter Plot (PCA Projection)')
sns.barplot(data=cluster_profile, x='Cluster', y='Average_Employment_Rate', hue='Cluster', palette='Oranges', legend=False, ax=axes[1, 0])
axes[1, 0].set_title(f'Cluster Profile (Silhouette = {best_score:.3f})')
for idx, row in cluster_profile.iterrows():
    axes[1, 0].text(idx, row['Average_Employment_Rate'] + 0.1, f"Risk={row['Risk_Rate']:.2f}", ha='center', fontsize=9)
dendrogram(linkage_matrix, truncate_mode='lastp', p=12, ax=axes[1, 1], color_threshold=None)
axes[1, 1].set_title(f'Dendrogram (Sample of {sample_size} records)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'cluster_analysis_dashboard.png', bbox_inches='tight')
plt.show()


C:\Users\user\AppData\Local\Temp\ipykernel_9140\3249181213.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_9140\3249181213.py:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_9140\3249181213.py:97: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## **Model Improvement**

This section improves the strongest baseline models using cross-validated hyperparameter tuning. It compares the tuned versions against the best baseline classifier and regressor so we can check whether the extra optimization actually improves predictive performance.

In [98]:
# Hyperparameter tuning for stronger predictive performance.
X_train_cls_tune, X_test_cls_tune, y_train_cls_tune, y_test_cls_tune = train_test_split(
    X, y_classification, test_size=0.2, random_state=RANDOM_STATE, stratify=y_classification
)
X_train_reg_tune, X_test_reg_tune, y_train_reg_tune, y_test_reg_tune = train_test_split(
    X, y_regression, test_size=0.2, random_state=RANDOM_STATE
)

best_baseline_classifier = classification_results.iloc[0]['Model']
best_baseline_regressor = regression_results.iloc[0]['Model']

classifier_search_space = {
    'Logistic Regression': (
        LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE),
        {'model__C': [0.5, 1.0, 5.0], 'model__solver': ['lbfgs'], 'model__max_iter': [3000]}
    ),
    'Random Forest': (
        RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=1),
        {'model__n_estimators': [300], 'model__max_depth': [None, 12], 'model__min_samples_leaf': [1, 3]}
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {'model__n_estimators': [100, 160], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [2, 3]}
    )
}

regressor_search_space = {
    'Ridge Regression': (
        Ridge(),
        {'model__alpha': [0.1, 1.0, 5.0]}
    ),
    'Random Forest Regressor': (
        RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
        {'model__n_estimators': [300], 'model__max_depth': [None, 12], 'model__min_samples_leaf': [1, 3]}
    ),
    'Gradient Boosting Regressor': (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {'model__n_estimators': [100, 160], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [2, 3]}
    )
}

best_cls_estimator, best_cls_grid = classifier_search_space[best_baseline_classifier]
best_reg_estimator, best_reg_grid = regressor_search_space[best_baseline_regressor]

classification_search = GridSearchCV(
    Pipeline(steps=[('preprocessor', build_preprocessor(X_train_cls_tune)), ('model', best_cls_estimator)]),
    param_grid=best_cls_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=1,
    refit=True
)
classification_search.fit(X_train_cls_tune, y_train_cls_tune)
tuned_cls_pred = classification_search.predict(X_test_cls_tune)
tuned_cls_score = classification_search.predict_proba(X_test_cls_tune)[:, 1] if hasattr(classification_search, 'predict_proba') else classification_search.decision_function(X_test_cls_tune)

regression_search = GridSearchCV(
    Pipeline(steps=[('preprocessor', build_preprocessor(X_train_reg_tune)), ('model', best_reg_estimator)]),
    param_grid=best_reg_grid,
    scoring='neg_root_mean_squared_error',
    cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=1,
    refit=True
)
regression_search.fit(X_train_reg_tune, y_train_reg_tune)
tuned_reg_pred = regression_search.predict(X_test_reg_tune)

tuned_classification_results = pd.DataFrame([
    {
        'Model': f'Tuned {best_baseline_classifier}',
        'Accuracy': accuracy_score(y_test_cls_tune, tuned_cls_pred),
        'Precision': precision_score(y_test_cls_tune, tuned_cls_pred, zero_division=0),
        'Recall': recall_score(y_test_cls_tune, tuned_cls_pred, zero_division=0),
        'F1': f1_score(y_test_cls_tune, tuned_cls_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test_cls_tune, tuned_cls_score)
    }
])

tuned_regression_results = pd.DataFrame([
    {
        'Model': f'Tuned {best_baseline_regressor}',
        'R2': r2_score(y_test_reg_tune, tuned_reg_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test_reg_tune, tuned_reg_pred)),
        'MAE': mean_absolute_error(y_test_reg_tune, tuned_reg_pred)
    }
])

classification_improvement = pd.concat([
    classification_results.head(1).assign(Type='Baseline'),
    tuned_classification_results.assign(Type='Tuned')
], ignore_index=True)
regression_improvement = pd.concat([
    regression_results.head(1).assign(Type='Baseline'),
    tuned_regression_results.assign(Type='Tuned')
], ignore_index=True)

print('Best tuned classification parameters:')
print(classification_search.best_params_)
print('\nBest tuned regression parameters:')
print(regression_search.best_params_)
print('\nClassification baseline vs tuned:')
print(classification_improvement.round(4).to_string(index=False))
print('\nRegression baseline vs tuned:')
print(regression_improvement.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=classification_improvement, x='Type', y='ROC_AUC', hue='Model', ax=axes[0])
axes[0].set_title('Baseline vs Tuned Classification ROC-AUC')
axes[0].set_xlabel('Model Version')
axes[0].set_ylabel('ROC-AUC')

sns.barplot(data=regression_improvement, x='Type', y='RMSE', hue='Model', ax=axes[1])
axes[1].set_title('Baseline vs Tuned Regression RMSE')
axes[1].set_xlabel('Model Version')
axes[1].set_ylabel('RMSE')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'model_tuning_comparison.png', bbox_inches='tight')
plt.show()

classification_improvement


Best tuned classification parameters:
{'model__C': 0.5, 'model__max_iter': 3000, 'model__solver': 'lbfgs'}

Best tuned regression parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__n_estimators': 160}

Classification baseline vs tuned:
                    Model  Accuracy  Precision  Recall     F1  ROC_AUC     Type
      Logistic Regression    0.7071     0.4551  0.8686 0.5972   0.8008 Baseline
Tuned Logistic Regression    0.7057     0.4551  0.8971 0.6038   0.8013    Tuned

Regression baseline vs tuned:
                            Model     R2   RMSE    MAE     Type
      Gradient Boosting Regressor 0.4517 4.6703 3.7183 Baseline
Tuned Gradient Boosting Regressor 0.4552 4.6556 3.7074    Tuned


C:\Users\user\AppData\Local\Temp\ipykernel_9140\4282830500.py:119: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Type
0,Logistic Regression,0.707143,0.455090,0.868571,0.597250,0.800827,Baseline
1,Tuned Logistic Regression,0.705714,0.455072,0.897143,0.603846,0.801328,Tuned


## **Advanced Ensemble Model**

Prompt engineering is mainly useful for language models, so for this tabular prediction task the closest equivalent is stronger model design. This section builds an advanced stacked ensemble for classification and regression, then compares it with the baseline and tuned models to check whether the ensemble improves prediction quality.

In [99]:
# Advanced stacked ensemble models.
X_train_cls_adv, X_test_cls_adv, y_train_cls_adv, y_test_cls_adv = train_test_split(
    X, y_classification, test_size=0.2, random_state=RANDOM_STATE, stratify=y_classification
)
X_train_reg_adv, X_test_reg_adv, y_train_reg_adv, y_test_reg_adv = train_test_split(
    X, y_regression, test_size=0.2, random_state=RANDOM_STATE
)

advanced_classifier = Pipeline(steps=[
    ('preprocessor', build_preprocessor(X_train_cls_adv)),
    ('model', StackingClassifier(
        estimators=[
            ('lr', LogisticRegression(C=0.5, max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE)),
            ('rf', RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=1)),
            ('gb', GradientBoostingClassifier(n_estimators=160, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE))
        ],
        final_estimator=LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE),
        cv=3,
        n_jobs=1
    ))
])
advanced_classifier.fit(X_train_cls_adv, y_train_cls_adv)
advanced_cls_pred = advanced_classifier.predict(X_test_cls_adv)
advanced_cls_score = advanced_classifier.predict_proba(X_test_cls_adv)[:, 1]

advanced_regressor = Pipeline(steps=[
    ('preprocessor', build_preprocessor(X_train_reg_adv)),
    ('model', StackingRegressor(
        estimators=[
            ('ridge', Ridge(alpha=1.0)),
            ('rf', RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1)),
            ('gb', GradientBoostingRegressor(n_estimators=160, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE))
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=3,
        n_jobs=1
    ))
])
advanced_regressor.fit(X_train_reg_adv, y_train_reg_adv)
advanced_reg_pred = advanced_regressor.predict(X_test_reg_adv)

advanced_classification_results = pd.DataFrame([
    {
        'Model': 'Advanced Stacking Classifier',
        'Accuracy': accuracy_score(y_test_cls_adv, advanced_cls_pred),
        'Precision': precision_score(y_test_cls_adv, advanced_cls_pred, zero_division=0),
        'Recall': recall_score(y_test_cls_adv, advanced_cls_pred, zero_division=0),
        'F1': f1_score(y_test_cls_adv, advanced_cls_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test_cls_adv, advanced_cls_score)
    }
])

advanced_regression_results = pd.DataFrame([
    {
        'Model': 'Advanced Stacking Regressor',
        'R2': r2_score(y_test_reg_adv, advanced_reg_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test_reg_adv, advanced_reg_pred)),
        'MAE': mean_absolute_error(y_test_reg_adv, advanced_reg_pred)
    }
])

classification_advanced_comparison = pd.concat([
    classification_results.head(1).assign(Type='Baseline'),
    tuned_classification_results.assign(Type='Tuned'),
    advanced_classification_results.assign(Type='Advanced')
], ignore_index=True)

regression_advanced_comparison = pd.concat([
    regression_results.head(1).assign(Type='Baseline'),
    tuned_regression_results.assign(Type='Tuned'),
    advanced_regression_results.assign(Type='Advanced')
], ignore_index=True)

print('Advanced classification model results:')
print(advanced_classification_results.round(4).to_string(index=False))
print('\nAdvanced regression model results:')
print(advanced_regression_results.round(4).to_string(index=False))
print('\nClassification comparison across baseline, tuned, and advanced models:')
print(classification_advanced_comparison.round(4).to_string(index=False))
print('\nRegression comparison across baseline, tuned, and advanced models:')
print(regression_advanced_comparison.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=classification_advanced_comparison, x='Type', y='ROC_AUC', hue='Model', ax=axes[0])
axes[0].set_title('Classification: Baseline vs Tuned vs Advanced')
axes[0].set_xlabel('Model Stage')
axes[0].set_ylabel('ROC-AUC')

sns.barplot(data=regression_advanced_comparison, x='Type', y='RMSE', hue='Model', ax=axes[1])
axes[1].set_title('Regression: Baseline vs Tuned vs Advanced')
axes[1].set_xlabel('Model Stage')
axes[1].set_ylabel('RMSE')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'advanced_model_comparison.png', bbox_inches='tight')
plt.show()

classification_advanced_comparison


Advanced classification model results:
                       Model  Accuracy  Precision  Recall     F1  ROC_AUC
Advanced Stacking Classifier    0.6886     0.4414  0.9257 0.5978   0.7926

Advanced regression model results:
                      Model     R2   RMSE    MAE
Advanced Stacking Regressor 0.4515 4.6711 3.7055

Classification comparison across baseline, tuned, and advanced models:
                       Model  Accuracy  Precision  Recall     F1  ROC_AUC     Type
         Logistic Regression    0.7071     0.4551  0.8686 0.5972   0.8008 Baseline
   Tuned Logistic Regression    0.7057     0.4551  0.8971 0.6038   0.8013    Tuned
Advanced Stacking Classifier    0.6886     0.4414  0.9257 0.5978   0.7926 Advanced

Regression comparison across baseline, tuned, and advanced models:
                            Model     R2   RMSE    MAE     Type
      Gradient Boosting Regressor 0.4517 4.6703 3.7183 Baseline
Tuned Gradient Boosting Regressor 0.4552 4.6556 3.7074    Tuned
      Advanced 

C:\Users\user\AppData\Local\Temp\ipykernel_9140\4162601792.py:96: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Type
0,Logistic Regression,0.707143,0.455090,0.868571,0.597250,0.800827,Baseline
1,Tuned Logistic Regression,0.705714,0.455072,0.897143,0.603846,0.801328,Tuned
2,Advanced Stacking Classifier,0.688571,0.441417,0.925714,0.597786,0.792631,Advanced


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tuned_results = []

for model_name, (model, param_grid) in classifier_search_space.items():

    pipeline = Pipeline(steps=[
        ('preprocessor', build_preprocessor(X_train_cls_tune)),
        ('model', model)
    ])

    search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train_cls_tune, y_train_cls_tune)

    y_pred = search.predict(X_test_cls_tune)
    y_prob = search.predict_proba(X_test_cls_tune)[:, 1]

    tuned_results.append({
        'Model': model_name,
        'Best_Params': search.best_params_,
        'ROC_AUC': roc_auc_score(y_test_cls_tune, y_prob),
        'Accuracy': accuracy_score(y_test_cls_tune, y_pred),
        'Precision': precision_score(y_test_cls_tune, y_pred, zero_division=0),
        'Recall': recall_score(y_test_cls_tune, y_pred, zero_division=0),
        'F1': f1_score(y_test_cls_tune, y_pred, zero_division=0)
    })

## Create comparison `table`

In [ ]:
tuned_results_df = pd.DataFrame(tuned_results)
tuned_results_df = tuned_results_df.sort_values(by='ROC_AUC', ascending=False)

print(tuned_results_df)

### Select best model

best_model_name = tuned_results_df.iloc[0]['Model']
print("Best model:", best_model_name)

In [ ]:
best_models = {}
best_models[model_name] = search.best_estimator_
best_model_name = tuned_results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

In [ ]:
y_pred = best_model.predict(X_test_cls_tune)
y_prob = best_model.predict_proba(X_test_cls_tune)[:, 1]

In [ ]:
predictions_df =X_test_cls_tune.copy()

predictions_df['Actual'] = y_test_cls_tune.values
predictions_df['Predicted'] = y_pred
predictions_df['Probability'] = y_prob

In [ ]:
predictions_df['Risk_Category'] = pd.cut(
    predictions_df['Probability'],
    bins=[0, 0.4, 0.7, 1],
    labels=['Low Employability', 'Moderate', 'High Employability']
)

In [ ]:
predictions_df.head()

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test_cls_tune, y_pred)
print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_cls_tune, y_pred)
print(cm)

In [ ]:
from pathlib import Path

output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)

predictions_df.to_csv(output_path / "model_predictions.csv", index=False)